# 英語のデータだけで学習して日本語を分類する (ゼロショットの言語間転移)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/crosslingual_transfer.ipynb)

NLP クックブック 12-20 ／ テキスト分類

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

日本語のラベル付きデータがないが、同じタスクの英語のデータならたくさんある。多言語モデルを英語のデータだけで微調整して、日本語の分類にそのまま使いたい。自然言語推論 (NLI、含意・中立・矛盾の 3 クラス) を例に、英語のデータを機械翻訳して学習した場合 (translate-train)、日本語のデータで学習した場合、日本語のデータを少しだけ足した場合 (few-shot) と比べる。

In [1]:
import csv, io, urllib.request, zipfile
import pandas as pd

LABELS = ["entailment", "neutral", "contradiction"]   # 3 つのデータで共通のラベル (含意・中立・矛盾)
N_TRAIN = 12000                                       # 各設定の学習件数 (1 エポック)

# 英語: SNLI (Bowman et al. 2015、CC BY-SA 4.0)。画像キャプションを前提文にして、仮説文をクラウドソーシングで書いたもの
hf = "https://huggingface.co/datasets/stanfordnlp/snli/resolve/main/plain_text/"
snli, snli_dev = [d[d["label"] >= 0].assign(label=lambda d: d["label"].map(dict(enumerate(LABELS))))   # -1 はラベルが割れたもの
                  for d in (pd.read_parquet(hf + f"{s}-00000-of-00001.parquet") for s in ["train", "validation"])]

# 英語を機械翻訳した日本語: JSNLI v1.1 (京都大学 黒橋研、CC BY-SA 4.0)。SNLI を機械翻訳したもの。dev はクラウドソーシングで翻訳の質とラベルを確かめてある
url = "https://nlp.ist.i.kyoto-u.ac.jp/DLcounter/lime.cgi?down=https://nlp.ist.i.kyoto-u.ac.jp/nl-resource/JSNLI/jsnli_1.1.zip&name=JSNLI.zip"
with urllib.request.urlopen(url) as r:
    z = zipfile.ZipFile(io.BytesIO(r.read()))
def read_jsnli(name):
    d = pd.read_csv(z.open(f"jsnli_1.1/{name}"), sep="\t", names=["label", "premise", "hypothesis"], quoting=csv.QUOTE_NONE)
    return d.assign(premise=d["premise"].str.replace(" ", ""), hypothesis=d["hypothesis"].str.replace(" ", ""))   # 分かち書きの空白を除く
jsnli, jsnli_dev = read_jsnli("train_w_filtering.tsv"), read_jsnli("dev.tsv")

# 日本語: JGLUE の JNLI v1.3 (ヤフーと早稲田大学、CC BY-SA 4.0)。日本語の画像キャプション (YJ Captions) から、SNLI とは独立に作ったもの
gh = "https://raw.githubusercontent.com/yahoojapan/JGLUE/main/datasets/jnli-v1.3/"
jnli, jnli_test = [pd.read_json(gh + f"{s}-v1.3.json", lines=True).rename(columns={"sentence1": "premise", "sentence2": "hypothesis"})
                  for s in ["train", "test"]]

for name, d in [("SNLI", snli), ("JSNLI", jsnli), ("JNLI", jnli), ("JNLI test", jnli_test)]:
    print(f"{name:8} {len(d):7} 件  ラベルの割合:", d["label"].value_counts(normalize=True).round(2).to_dict())
print(jsnli[["premise", "hypothesis", "label"]].head(2).to_string(index=False))
print(jnli[["premise", "hypothesis", "label"]].head(2).to_string(index=False))

SNLI      549367 件  ラベルの割合: {'entailment': 0.33, 'contradiction': 0.33, 'neutral': 0.33}
JSNLI     533005 件  ラベルの割合: {'contradiction': 0.34, 'neutral': 0.33, 'entailment': 0.33}
JNLI       20073 件  ラベルの割合: {'neutral': 0.56, 'contradiction': 0.3, 'entailment': 0.14}
JNLI test    2508 件  ラベルの割合: {'neutral': 0.54, 'contradiction': 0.31, 'entailment': 0.15}
                premise                     hypothesis         label
      ガレージで、壁にナイフを投げる男。 男は魔法のショーのためにナイフを投げる行為を練習しています。       neutral
茶色のドレスを着た女性がベンチに座っています。                   女性が畑で踊っています。 contradiction
               premise             hypothesis   label
二人の男性がジャンボジェット機を見ています。   2人の男性が、白い飛行機を眺めています。 neutral
  2人の男性が、白い飛行機を眺めています。 二人の男性がジャンボジェット機を見ています。 neutral


In [2]:
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from tokenizers import normalizers
from transformers.utils import logging

logging.set_verbosity_error()   # 分類用の層を新しく初期化したという報告を出さない

MODEL = "microsoft/mdeberta-v3-base"   # DeBERTaV3 の多言語版。XLM-R と同じ CC100 で事前学習 (MIT ライセンス)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
# transformers 5 系の DebertaV2Tokenizer は、sentencepiece の NFKC 正規化を落とし、全角の記号や英数字 (（、ＡＢＣ、１２３ など) を
# [UNK] にしてしまう (issue #49622)。このモデルには tokenizer.json がないので、正規化の先頭に NFKC を足して元の spm.model の分割にそろえる
tokenizer.backend_tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC(), tokenizer.backend_tokenizer.normalizer])
print(tokenizer.tokenize("Ｔシャツを着た２人の子ども（左）"))   # [UNK] が出ないことを確かめる
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う

def encode(df):
    ds = Dataset.from_dict({"premise": list(df["premise"]), "hypothesis": list(df["hypothesis"]),
                            "label": [LABELS.index(l) for l in df["label"]]})
    return ds.map(lambda b: tokenizer(b["premise"], b["hypothesis"], truncation=True, max_length=128),
                  batched=True, remove_columns=["premise", "hypothesis"])

def train(df, init=MODEL, epochs=1, lr=3e-5):
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(
        # 重みが float16 で保存されていて (212 個中 210 個)、既定では float16 のまま読まれるので、float32 で読む
        # (fp16 の混合精度の学習には float32 の重みが必要)
        init, dtype=torch.float32, num_labels=3,
        id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})
    args = TrainingArguments(output_dir="out", num_train_epochs=epochs, learning_rate=lr, weight_decay=0.01,
                             warmup_steps=0.1,   # 1 未満は全ステップに対する割合
                             per_device_train_batch_size=32, per_device_eval_batch_size=128, save_strategy="no",
                             logging_strategy="no", report_to="none", bf16=BF16, fp16=not BF16, seed=0)
    trainer = Trainer(model=model, args=args, train_dataset=encode(df), data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    return trainer

EVAL = {"SNLI dev (英語)": snli_dev.sample(2000, random_state=0), "JSNLI dev (SNLI の翻訳)": jsnli_dev, "JNLI test (日本語)": jnli_test}
EVAL_DS = {k: encode(d) for k, d in EVAL.items()}

JNLI_PRED = {}   # 設定ごとの JNLI test の予測 (後でブートストラップと混同行列に使う)

def evaluate(trainer, setting=None):
    row, preds = {}, {}
    for name, ds in EVAL_DS.items():
        preds[name] = trainer.predict(ds).predictions.argmax(-1)
        row[name] = round(accuracy_score(ds["label"], preds[name]), 3)
    gold, pred = EVAL_DS["JNLI test (日本語)"]["label"], preds["JNLI test (日本語)"]
    row["JNLI macro-F1"] = round(f1_score(gold, pred, average="macro"), 3)
    row["JNLI の予測の割合 (含意/中立/矛盾)"] = "/".join(f"{x:.2f}" for x in np.bincount(pred, minlength=3) / len(pred))
    if setting:
        JNLI_PRED[setting] = pred
    return row

print("JNLI test の多数派 (中立) を答え続けたときの正解率:", round((jnli_test["label"] == "neutral").mean(), 3))

['▁T', 'シャツ', 'を着', 'た', '2', '人の', '子ども', '(', '左', ')']
JNLI test の多数派 (中立) を答え続けたときの正解率: 0.544


In [3]:
# 3 つの設定で学習して比べる (Colab の T4 で合計 6〜8 分の見込み)
results = {}
trainer_en = train(snli.sample(N_TRAIN, random_state=0))
results["英語 (SNLI) で学習 = ゼロショット転移"] = evaluate(trainer_en, "zero-shot")
trainer_en.save_model("mdeberta-snli")                     # 後で few-shot の出発点にする
results["機械翻訳 (JSNLI) で学習 = translate-train"] = evaluate(train(jsnli.sample(N_TRAIN, random_state=0)), "translate-train")
results["日本語 (JNLI) で学習"] = evaluate(train(jnli.sample(N_TRAIN, random_state=0)))
pd.set_option("display.width", 200)
pd.DataFrame(results).T

{'train_runtime': '39.77', 'train_samples_per_second': '301.7', 'train_steps_per_second': '9.429', 'train_loss': '0.679', 'epoch': '1'}
{'train_runtime': '40.84', 'train_samples_per_second': '293.8', 'train_steps_per_second': '9.182', 'train_loss': '0.7577', 'epoch': '1'}
{'train_runtime': '35.77', 'train_samples_per_second': '335.5', 'train_steps_per_second': '10.48', 'train_loss': '0.489', 'epoch': '1'}


,SNLI dev (英語),JSNLI dev (SNLI の翻訳),JNLI test (日本語),JNLI macro-F1,JNLI の予測の割合 (含意/中立/矛盾)
英語 (SNLI) で学習 = ゼロショット転移,0.863,0.849,0.634,0.625,0.36/0.32/0.32
機械翻訳 (JSNLI) で学習 = translate-train,0.838,0.858,0.673,0.658,0.29/0.37/0.34
日本語 (JNLI) で学習,0.535,0.645,0.893,0.86,0.12/0.57/0.31


In [4]:
# JNLI test の 2508 件をブートストラップして 95% 信頼区間を出す。評価データの抽出によるばらつきだけで、乱数の種によるばらつきは含まない
gold = np.array(EVAL_DS["JNLI test (日本語)"]["label"])
c_zs, c_tt = [(JNLI_PRED[k] == gold).astype(float) for k in ["zero-shot", "translate-train"]]   # 正解なら 1
rng = np.random.default_rng(0)
idx = rng.integers(0, len(gold), (1000, len(gold)))   # 2 つの設定で同じ組を引き直す (対応のあるブートストラップ)
lo, hi = np.percentile(c_zs[idx].mean(1), [2.5, 97.5])
print(f"ゼロショット転移の JNLI 正解率 {c_zs.mean():.3f} (95% 信頼区間 {lo:.3f}〜{hi:.3f})")
lo, hi = np.percentile((c_tt[idx] - c_zs[idx]).mean(1), [2.5, 97.5])
print(f"translate-train − ゼロショット転移 = {c_tt.mean() - c_zs.mean():+.3f} (95% 信頼区間 {lo:+.3f}〜{hi:+.3f})")

# どのラベルを取り違えているか (行が正解、列が予測)
from sklearn.metrics import confusion_matrix
print(pd.DataFrame(confusion_matrix(gold, JNLI_PRED["zero-shot"]), index=LABELS, columns=LABELS))

ゼロショット転移の JNLI 正解率 0.634 (95% 信頼区間 0.615〜0.654)
translate-train − ゼロショット転移 = +0.039 (95% 信頼区間 +0.024〜+0.055)
               entailment  neutral  contradiction
entailment            330       26             11
neutral               430      705            230
contradiction         142       78            556


In [5]:
# few-shot: 英語で学習したモデルに、日本語の学習データを少しだけ足して続けて学習する (Colab の T4 で 2〜3 分の見込み)
for k in [100, 1000]:
    few = jnli.sample(k, random_state=1)
    epochs = 10 if k == 100 else 3
    r_cont = evaluate(train(few, init="mdeberta-snli", epochs=epochs))     # 英語 → 日本語 k 件
    r_only = evaluate(train(few, epochs=epochs))                         # 日本語 k 件だけ (英語なし)
    print(f"JNLI {k:4} 件: 英語で学習してから追加 → 正解率 {r_cont['JNLI test (日本語)']:.3f} / macro-F1 {r_cont['JNLI macro-F1']:.3f}"
          f"   日本語だけで学習 → 正解率 {r_only['JNLI test (日本語)']:.3f} / macro-F1 {r_only['JNLI macro-F1']:.3f}")

{'train_runtime': '3.731', 'train_samples_per_second': '268', 'train_steps_per_second': '10.72', 'train_loss': '0.2087', 'epoch': '10'}
{'train_runtime': '3.388', 'train_samples_per_second': '295.2', 'train_steps_per_second': '11.81', 'train_loss': '0.8291', 'epoch': '10'}
JNLI  100 件: 英語で学習してから追加 → 正解率 0.813 / macro-F1 0.775   日本語だけで学習 → 正解率 0.690 / macro-F1 0.485
{'train_runtime': '9.382', 'train_samples_per_second': '319.7', 'train_steps_per_second': '10.23', 'train_loss': '0.38', 'epoch': '3'}
{'train_runtime': '8.868', 'train_samples_per_second': '338.3', 'train_steps_per_second': '10.82', 'train_loss': '0.7396', 'epoch': '3'}
JNLI 1000 件: 英語で学習してから追加 → 正解率 0.860 / macro-F1 0.831   日本語だけで学習 → 正解率 0.756 / macro-F1 0.578


In [6]:
# 英語だけで学習したモデルに、日本語の文の組を入力してみる
from transformers import pipeline

clf = pipeline("text-classification", model="mdeberta-snli", tokenizer=tokenizer, device=0)
pairs = [("公園で子どもたちがサッカーをしている。", "子どもたちが外で遊んでいる。"),
         ("公園で子どもたちがサッカーをしている。", "子どもたちは教室で授業を受けている。"),
         ("公園で子どもたちがサッカーをしている。", "子どもたちは試合に勝った。")]
for p, h in pairs:
    r = clf({"text": p, "text_pair": h})
    print(f"{r['label']:13} {r['score']:.2f}  {p} → {h}")

entailment    0.95  公園で子どもたちがサッカーをしている。 → 子どもたちが外で遊んでいる。
contradiction 0.96  公園で子どもたちがサッカーをしている。 → 子どもたちは教室で授業を受けている。
neutral       0.90  公園で子どもたちがサッカーをしている。 → 子どもたちは試合に勝った。


- 多言語モデルは、英語のデータだけで微調整しても日本語の入力をある程度扱える。手順はこの章の「BERT をファインチューニングして文書分類する (Trainer)」と同じで、モデルを多言語のものに替えるだけでよい。
- 上の表では、3 つの比較で言語の効果とデータセットの効果を切り分けている。(1) 言語だけが違う: 英語で学習したモデルは SNLI dev で 0.863、JSNLI dev で 0.849 で、差は 1.4 ポイントだった。(2) 言語をそろえても作り方が違う: 機械翻訳した JSNLI で学習しても JNLI では 0.673 で、JNLI で学習した場合 (0.893) に遠い。(3) 逆向き: JNLI で学習したモデルは SNLI dev で 0.535、JSNLI dev で 0.645 になる。translate-train のモデルも、英語の SNLI dev では 0.838 と下がる。SNLI (Flickr30k) も JNLI (YJ Captions) も画像キャプションが元で分野は近いので、JNLI との差の大部分は、ラベルの付け方の違いによるものと考えられる。なお、translate-train の学習データは機械翻訳した翻訳調の日本語である。
- JSNLI dev は、SNLI の dev を機械翻訳し、日本語として意味が通るか、ラベルが正しいかをクラウドソーシングで確かめて絞り込んだ 3,916 組である (JSNLI の README)。SNLI dev より易しい組に偏っている可能性があり、1.4 ポイントは言語の効果を小さく見せているかもしれない。また機械翻訳の文なので、JSNLI dev での translate-train (0.858) とゼロショット転移 (0.849) の比較は、translate-train に有利である。
- JNLI では、ゼロショット転移の正解率は 0.634 (95% 信頼区間 0.615〜0.654) で、多数派のクラスを答え続けた場合 (0.544) は上回る。translate-train との差は +0.039 (対応のあるブートストラップで 95% 信頼区間 +0.024〜+0.055) だった。この区間は評価データの抽出によるばらつきだけで、乱数の種によるばらつきは含まない。種 1 つ、1 回の学習の結果なので、複数の種で確かめる。
- 混同行列を見ると、英語で学習したモデルは JNLI の中立 1365 件のうち 430 件を含意と判定している。ラベルの割合も、SNLI はほぼ均等なのに対し、JNLI は中立が半分を超える。言語間転移を試す前に、元のデータと使いたいデータで、ラベルの定義や割合がそろっているかを確かめる。
- few-shot は、この設定 (学習率・エポック数は調整していない、種 1 つ) では、英語で学習したモデルに JNLI の 100 件を足すと正解率 0.813、1000 件で 0.860 になり、JNLI の 12000 件で学習した場合 (0.893) に近づいた。同じ件数の日本語だけで学習すると 0.690 と 0.756 だった。学習データが小さいほど (特に 100 件では) 種によって結果が大きく変わるので、複数の種で平均をとる。
- mDeBERTa-v3-base のモデルカードでは、英語のデータだけで学習して XNLI の 15 言語 (英語を含む) の dev で評価した正解率の平均が、XLM-R base より高い (79.8 と 76.2)。この章の「ラベル付きデータなしで分類する (ゼロショット分類、NLI)」で使った `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` も、このモデルを MNLI と XNLI (日本語を含まない 15 言語) で微調整したもので、日本語への適用は言語間転移にあたる。
- 検索の章の「言語をまたいで検索する (クロスリンガル検索)」は多言語の埋め込みで言語をまたいだ。ここでは多言語の事前学習モデルを微調整した分類器で言語をまたいでいる。データは SNLI、JSNLI、JNLI のいずれも CC BY-SA 4.0、モデルは MIT ライセンス。CC BY-SA の継承の条件が学習したモデルにかかるかは確定していないが、かかる可能性があるので、配布するときは注意する。